Используя базу "Аудиожанры", примените подход к музыке как к тексту и напишите сверточный классификатор (на базе слоя Conv1D) для подготовленных данных. Для этого:

1. Измените подготовку данных так, чтобы набор признаков, извлекаемый из аудиофайла, был представлен в виде последовательностей векторов признаков. Последовательности должны быть фиксированного размера и выбираться скользящим окном c заданным шагом. Другими словами: берем аудио-файл длительность, например, 30 сек. Берем отрезок фиксированной длины (например, 5с) и получаем набор признаков для этого отрезка. Смещаемся на шаг (например, 1с) и берем следующий отрезок. Таким образом готовим обучающую вборку.
2. Длину последовательности, размер шага и достаточный набор признаков определите самостоятельно исходя из требований к точности классификатора;
3. Разработайте классификатор на одномерных сверточных слоях **Conv1D** с точностью классификации жанра на тестовых данных не ниже **60%**, а на обучающих файлах - **68%** и выше;
4. Используйте за основу материал с урока, но при желании разработайте свои инструменты.

In [ ]:
import os
import time
import warnings

import gdown
import librosa
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')

%matplotlib inline

print('Библиотеки загружены')

In [ ]:
gdown.download(
    'https://storage.yandexcloud.net/aiueducation/Content/base/l12/genres.zip',
    None,
    quiet=True
)

!unzip -qo genres.zip

print('Список доступных жанров:')
!ls genres

In [ ]:
DATA_DIR = './genres'
GENRE_NAMES = sorted(os.listdir(DATA_DIR))
GENRE_COUNT = len(GENRE_NAMES)

TRACK_SECONDS = 30
FRAME_COUNT = 1200
WINDOW_LENGTH = 150
WINDOW_STEP = 50


def extract_audio_table(path, duration=TRACK_SECONDS):
    signal, rate = librosa.load(path, mono=True, duration=duration)

    fft_size = 2048
    hop_size = 512

    chroma = librosa.feature.chroma_stft(y=signal, sr=rate, n_fft=fft_size, hop_length=hop_size)
    mfcc = librosa.feature.mfcc(y=signal, sr=rate, n_fft=fft_size, hop_length=hop_size, n_mfcc=20)
    rms = librosa.feature.rms(y=signal, hop_length=hop_size)
    centroid = librosa.feature.spectral_centroid(y=signal, sr=rate, n_fft=fft_size, hop_length=hop_size)
    bandwidth = librosa.feature.spectral_bandwidth(y=signal, sr=rate, n_fft=fft_size, hop_length=hop_size)
    rolloff = librosa.feature.spectral_rolloff(y=signal, sr=rate, n_fft=fft_size, hop_length=hop_size)
    zero_rate = librosa.feature.zero_crossing_rate(signal, hop_length=hop_size)

    table = np.vstack([
        chroma,
        mfcc,
        rms,
        centroid,
        bandwidth,
        rolloff,
        zero_rate
    ]).T.astype('float32')

    if table.shape[0] < FRAME_COUNT:
        table = np.pad(table, ((0, FRAME_COUNT - table.shape[0]), (0, 0)), mode='constant')
    else:
        table = table[:FRAME_COUNT]

    return table


def split_by_window(table, length=WINDOW_LENGTH, step=WINDOW_STEP):
    result = []
    for start in range(0, table.shape[0] - length + 1, step):
        result.append(table[start:start + length])
    return np.asarray(result, dtype='float32')


def load_audio_tables(root_dir, genres, train_count=90):
    train_tables, train_labels = [], []
    test_tables, test_labels = [], []

    for genre_index, genre_name in enumerate(genres):
        folder = os.path.join(root_dir, genre_name)
        names = sorted(os.listdir(folder))

        for file_index, file_name in enumerate(names):
            file_path = os.path.join(folder, file_name)
            try:
                feature_table = extract_audio_table(file_path)
            except Exception:
                continue

            if file_index < train_count:
                train_tables.append(feature_table)
                train_labels.append(genre_index)
            else:
                test_tables.append(feature_table)
                test_labels.append(genre_index)

    return train_tables, train_labels, test_tables, test_labels


def scale_tables(train_tables, test_tables):
    scaler = StandardScaler()
    scaler.fit(np.concatenate(train_tables, axis=0))

    train_scaled = [scaler.transform(item).astype('float32') for item in train_tables]
    test_scaled = [scaler.transform(item).astype('float32') for item in test_tables]

    return train_scaled, test_scaled, scaler


def build_window_arrays(tables, labels):
    x_data, y_data = [], []

    for table, label in zip(tables, labels):
        windows = split_by_window(table)
        x_data.append(windows)
        y_data.append(np.full(windows.shape[0], label, dtype='int64'))

    x_data = np.concatenate(x_data, axis=0).astype('float32')
    y_data = np.concatenate(y_data, axis=0).astype('int64')

    return x_data, y_data

In [ ]:
print('Начинается извлечение признаков')
start_time = time.time()

train_raw, train_track_labels, test_raw, test_track_labels = load_audio_tables(DATA_DIR, GENRE_NAMES)

print(f'Извлечение завершено за {round(time.time() - start_time)} сек.')

train_scaled, test_scaled, audio_scaler = scale_tables(train_raw, test_raw)

X_train, y_train = build_window_arrays(train_scaled, train_track_labels)
X_test, y_test = build_window_arrays(test_scaled, test_track_labels)

print('Размеры массивов:')
print(f'X_train: {X_train.shape}, y_train: {y_train.shape}')
print(f'X_test:  {X_test.shape}, y_test:  {y_test.shape}')

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

BATCH_SIZE = 64
EPOCHS = 40
LEARNING_RATE = 0.0005

train_dataset = TensorDataset(
    torch.tensor(X_train, dtype=torch.float32),
    torch.tensor(y_train, dtype=torch.long)
)

test_dataset = TensorDataset(
    torch.tensor(X_test, dtype=torch.float32),
    torch.tensor(y_test, dtype=torch.long)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)


class AudioConvNet(nn.Module):
    def __init__(self, feature_count, class_count, sequence_length):
        super().__init__()

        self.features = nn.Sequential(
            nn.Conv1d(feature_count, 64, kernel_size=5, padding=2),
            nn.ReLU(),
            nn.BatchNorm1d(64),
            nn.MaxPool1d(2),
            nn.Dropout1d(0.2),

            nn.Conv1d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(128),
            nn.MaxPool1d(2),
            nn.Dropout1d(0.2),

            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.MaxPool1d(2),
            nn.Dropout1d(0.3)
        )

        with torch.no_grad():
            probe = torch.zeros(1, feature_count, sequence_length)
            flat_size = self.features(probe).reshape(1, -1).shape[1]

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(flat_size, 256),
            nn.ReLU(),
            nn.BatchNorm1d(256),
            nn.Dropout(0.4),
            nn.Linear(256, class_count)
        )

    def forward(self, x):
        x = x.permute(0, 2, 1)
        x = self.features(x)
        x = self.classifier(x)
        return x


model = AudioConvNet(
    feature_count=X_train.shape[2],
    class_count=GENRE_COUNT,
    sequence_length=X_train.shape[1]
).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

print(model)

In [ ]:
def run_epoch(model, loader, loss_fn, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total_items = 0

    with torch.set_grad_enabled(is_train):
        for batch_x, batch_y in loader:
            batch_x = batch_x.to(device)
            batch_y = batch_y.to(device)

            if is_train:
                optimizer.zero_grad()

            logits = model(batch_x)
            loss = loss_fn(logits, batch_y)

            if is_train:
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * batch_x.size(0)
            total_correct += (logits.argmax(1) == batch_y).sum().item()
            total_items += batch_x.size(0)

    return total_loss / total_items, total_correct / total_items


history = {
    'loss': [],
    'accuracy': [],
    'val_loss': [],
    'val_accuracy': []
}

print('Запуск обучения модели Conv1D на PyTorch')

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, criterion, optimizer)
    test_loss, test_acc = run_epoch(model, test_loader, criterion)

    history['loss'].append(train_loss)
    history['accuracy'].append(train_acc)
    history['val_loss'].append(test_loss)
    history['val_accuracy'].append(test_acc)

    print(
        f'Эпоха {epoch:02d}/{EPOCHS} | '
        f'loss: {train_loss:.4f} | accuracy: {train_acc:.4f} | '
        f'val_loss: {test_loss:.4f} | val_accuracy: {test_acc:.4f}'
    )

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 5))
fig.suptitle('Обучение Conv1D-классификатора жанров на PyTorch')

ax1.plot(history['accuracy'], label='Точность на обучении')
ax1.plot(history['val_accuracy'], label='Точность на тесте')
ax1.set_xlabel('Эпоха')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True)

ax2.plot(history['loss'], label='Ошибка на обучении')
ax2.plot(history['val_loss'], label='Ошибка на тесте')
ax2.set_xlabel('Эпоха')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True)

plt.show()

train_loss, train_acc = run_epoch(model, train_loader, criterion)
test_loss, test_acc = run_epoch(model, test_loader, criterion)

print('\n' + '=' * 50)
print(f'Итоговая точность на обучающих данных: {round(train_acc * 100, 2)}% (Требуется: >= 68%)')
print(f'Итоговая точность на тестовых данных:  {round(test_acc * 100, 2)}% (Требуется: >= 60%)')
print('=' * 50)

if train_acc >= 0.68 and test_acc >= 0.60:
    print('Требования Pro-уровня выполнены')
else:
    print('Целевая точность не достигнута')